# Gemma 2 9B — checkpoint-1440 submission (no full validation)

**LLM Classification Finetuning · inference only · existing checkpoint-1440 dataset.**

Your completed `3_arena_gemma2_9b_memory_safe(2).ipynb` still selected **checkpoint-1440**.
Its final recipe is **9B weight 1.0, 2B weight 0.0, A/B-swap TTA, temperature 0.9**.
The notebook below uses that fixed recipe, not a fresh hyperparameter search.

| Local result from the completed notebook | Selection log loss | Audit log loss | Full-fold log loss |
|---|---:|---:|---:|
| 9B, original order, T=1 | 0.895299 | 0.914849 | 0.905076 |
| 9B, A/B TTA, T=1 | 0.883663 | 0.898659 | 0.891163 |
| **Selected: 9B, A/B TTA, T=0.9** | **0.881159** | **0.898040** | **0.889601** |

Full-fold scores are descriptive after selection; they are not leaderboard scores.

## Run in Kaggle
1. Import **this notebook into a fresh session**. Do not append its cells to the old notebook.
2. Attach your existing private **`9b-1440`** dataset and the **LLM Classification Finetuning** competition data.
3. Select **GPU T4 ×2**, **Internet OFF**. Run **Save Version → Save & Run All**.
4. After the saved version succeeds, submit its `/kaggle/working/submission.csv` through the competition panel.

**No retraining, new exporter, new model upload, or 2B dataset is required.** The source model files are read-only.
The earlier dataset manifest describes an interim original-order recipe; this notebook deliberately overrides
only that prediction recipe with the result selected by your completed training run.

## What changed
The traceback reached `score.modules_to_save['default']` and then a copied Accelerate hook inside the head.
Moving inputs at the outer wrapper was insufficient: the inner hook moved them again.
This version removes that runtime head wrapper, builds a hook-free inference head from the **exact saved FP32
weight tensor**, and explicitly moves its input onto the weight's device inside its forward method.
LoRA modules also receive fresh device-alignment hooks after adapter loading. The backbone partition is unchanged.

T4 inference still uses FP16 backbone computation, whereas training used BF16. **Full validation is removed:**
the notebook does not load `train.csv`, infer on the selection set, or compare selection log loss.
The small head-routing and full-model smoke tests remain, followed directly by test prediction with A/B-swap TTA
and temperature 0.9. Smoke tests check execution and finite outputs, not BF16/FP16 score equivalence.

In [ ]:
import json
import os
import sys
import time
import shutil
import zipfile
from pathlib import Path

NOTEBOOK_STARTED = time.perf_counter()
ASSET_DIR = None  # Auto-detect the checkpoint-1440 private dataset.
COMPETITION_DIR = None
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 4  # Length-sorted inference; back off to 2/1 only after an actual CUDA OOM.
FINAL_RECIPE = {"checkpoint_step": 1440, "swap_tta": True,
                "temperature": 0.9, "weight_9b": 1.0, "weight_2b": 0.0}
NOTEBOOK_REVISION = "9b-1440-final-headfix-v3-no-validation"
print("RUNNING:", NOTEBOOK_REVISION)
# A failed interactive retry must not leave an old CSV looking like a new result.
(WORK_DIR / "submission.csv").unlink(missing_ok=True)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


if ASSET_DIR is None:
    matches = []
    for p in INPUT_ROOT.rglob("checkpoint_manifest.json"):
        candidate = read_json(p)
        if candidate.get("bundle_format") == "arena-gemma2-9b-checkpoint-v1" and candidate.get("checkpoint_step") == 1440:
            matches.append(p)
    if not matches:
        archives = list(INPUT_ROOT.rglob("arena_9b_checkpoint1440_assets.zip"))
        if len(archives) == 1:
            extract_dir = WORK_DIR / "_checkpoint1440_assets"
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as z:
                required = sum(m.file_size for m in z.infolist())
                if shutil.disk_usage(WORK_DIR).free < required + 1024**3:
                    raise RuntimeError("Insufficient disk to extract ZIP. Attach the dataset with files expanded.")
                for member in z.infolist():
                    target = (extract_dir / member.filename).resolve()
                    if not target.is_relative_to(extract_dir.resolve()):
                        raise ValueError("Unsafe ZIP path.")
                z.extractall(extract_dir)
            matches = list(extract_dir.rglob("checkpoint_manifest.json"))
    if len(matches) != 1:
        raise FileNotFoundError("Attach exactly one checkpoint-1440 bundle, or set ASSET_DIR explicitly.")
    ASSET_DIR = matches[0].parent
else:
    ASSET_DIR = Path(ASSET_DIR)
manifest = read_json(ASSET_DIR / "checkpoint_manifest.json")
assert manifest["bundle_format"] == "arena-gemma2-9b-checkpoint-v1"
assert manifest["checkpoint_step"] == 1440 and manifest["model_name"] == "google/gemma-2-9b-it"
assert manifest["competition"] == "llm-classification-finetuning"
source_bundle_recipe = manifest.get("inference_recipe")
# Keep the existing bundle unchanged. The final recipe above was selected AFTER it was exported.
BASE_DIR = ASSET_DIR / manifest["base_subdir"]
ADAPTER_DIR = ASSET_DIR / manifest["adapter_subdir"]
preprocessing = read_json(ASSET_DIR / "preprocessing.json")
print("Using:", ASSET_DIR)
print("Checkpoint:", manifest["checkpoint_step"], "| recorded selection log loss:", manifest["checkpoint_selection_log_loss"])
print("Final inference recipe:", FINAL_RECIPE)

## 1. Install the existing offline runtime
The same bundled Transformers/PEFT/Accelerate wheels are used. Kaggle's CUDA/PyTorch, NumPy, and pandas are not
replaced. Run in a fresh session so there are no imported copies of the previous model libraries.

In [ ]:
import importlib
import subprocess
from importlib import metadata

if any(name in sys.modules for name in ["transformers", "peft", "accelerate", "tokenizers"]):
    raise RuntimeError("Model libraries are already imported. Restart this Kaggle session and run from the top.")
py = f"{sys.version_info.major}.{sys.version_info.minor}"
if py not in manifest["runtimes"]:
    raise RuntimeError(f"Bundle supports Python {list(manifest['runtimes'])}, but Kaggle uses {py}. Re-export with that version.")
runtime = manifest["runtimes"][py]
wheel_dir = ASSET_DIR / runtime["wheel_subdir"]
runtime_dir = WORK_DIR / "_arena_9b_runtime_v3"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--no-index", "--find-links", str(wheel_dir), "--no-deps", "--target", str(runtime_dir),
    "--upgrade", "-r", str(wheel_dir / "requirements.txt")])
sys.path.insert(0, str(runtime_dir))
importlib.invalidate_caches()
from packaging.requirements import Requirement
for requirement in runtime["preserved_requirements"]:
    req = Requirement(requirement)
    if not req.specifier.contains(metadata.version(req.name), prereleases=True):
        raise RuntimeError(f"Kaggle's {req.name}=={metadata.version(req.name)} does not satisfy {requirement}.")
print("Offline runtime installed. Kaggle's existing CUDA/PyTorch was preserved.")

## 2. Verify the checkpoint bundle and read competition inputs
Hashes verify the exported weights, not a newly downloaded model. Only the competition's current `test.csv`
is loaded for prediction; no validation data or labels are loaded, and no row count is hardcoded.

In [ ]:
import gc
import hashlib
import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from safetensors import safe_open
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from tqdm.auto import tqdm

# Must happen BEFORE PEFT dispatches adapter modules. This model does not use TorchAO.
import peft.tuners.lora.torchao as peft_torchao
peft_torchao.is_torchao_available = lambda: False


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


for rel, info in tqdm(manifest["files"].items(), desc="Verify uploaded assets"):
    p = ASSET_DIR / rel
    assert p.is_file() and p.stat().st_size == info["size"], f"Missing/incomplete file: {rel}"
    assert sha256_file(p) == info["sha256"], f"Checksum mismatch: {rel}"
assert manifest["files"]["adapter/adapter_model.safetensors"]["sha256"] == manifest["adapter_sha256"]

if COMPETITION_DIR is None:
    default_dir = INPUT_ROOT / "llm-classification-finetuning"
    if (default_dir / "test.csv").is_file():
        COMPETITION_DIR = default_dir
    else:
        candidates = [p.parent for p in INPUT_ROOT.rglob("test.csv")
                      if "llm-classification-finetuning" in p.parts]
        if len(candidates) != 1:
            raise FileNotFoundError("Use Add Input to attach LLM Classification Finetuning competition data.")
        COMPETITION_DIR = candidates[0]
else:
    COMPETITION_DIR = Path(COMPETITION_DIR)
LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
LABEL_NAMES = ["win_a", "win_b", "tie"]
assert manifest["label_columns"] == LABEL_COLUMNS and manifest["label_names"] == LABEL_NAMES

test_frame = pd.read_csv(COMPETITION_DIR / "test.csv", dtype={"id": str})
assert len(test_frame) > 0 and test_frame["id"].is_unique
assert {"id", "prompt", "response_a", "response_b"}.issubset(test_frame.columns)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), local_files_only=True)
tokenizer.padding_side = "right"
assert all(x is not None for x in [tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id])
assert preprocessing["version"] == "full-if-fits-v2"
MAX_LENGTH, PROMPT_BUDGET = preprocessing["max_length"], preprocessing["prompt_budget"]
assert (MAX_LENGTH, PROMPT_BUDGET) == (2048, 384)
assert preprocessing["padding_side"] == "right" and preprocessing["label_names"] == LABEL_NAMES
prefix_ids, a_header_ids, b_header_ids, suffix_ids, omission_ids = [
    tokenizer.encode(preprocessing[key], add_special_tokens=False)
    for key in ["prefix", "a_header", "b_header", "suffix", "omission"]
]
overhead = sum(map(len, [prefix_ids, a_header_ids, b_header_ids, suffix_ids])) + 2
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
print(f"Test rows: {len(test_frame):,} | full validation disabled")
print({n:metadata.version(n) for n in ["torch", "transformers", "peft", "accelerate"]})

## 3. Preserve the exact 2,048-token training preprocessing
The parsing, turn formatting, allocation, and trimming functions below are unchanged from the uploaded 9B training
notebook. Original-order fingerprints must match its saved tokenization cases exactly. For TTA, exchange the two
responses **before** allocating tokens, and later exchange probability columns 0 and 1 back.

In [ ]:
def parse_turns(value):
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list) or not all(t is None or isinstance(t, str) for t in turns):
        raise ValueError("Each competition text field must decode to a list of strings/nulls.")
    return ["" if t is None else t for t in turns] or [""]


def format_turns(turns):
    return "\n\n".join(f"[Turn {i + 1}]\n{t if t else '[No text]'}" for i, t in enumerate(turns))


def trim_tokens(ids, budget):
    if len(ids) <= budget:
        return list(ids)
    if budget <= len(omission_ids):
        return list(ids[:budget])
    remain = budget - len(omission_ids)
    head, tail = (remain + 1) // 2, remain // 2
    return list(ids[:head]) + omission_ids + (list(ids[-tail:]) if tail else [])


def allocate_budgets(lengths):
    lp, la, lb = map(int, lengths)
    available = MAX_LENGTH - overhead
    if lp + la + lb <= available:
        return lp, la, lb  # The key fix: no unnecessary prompt truncation.
    p = min(lp, PROMPT_BUDGET)
    answer_space = available - p
    a, b = min(la, answer_space // 2), min(lb, answer_space // 2)
    spare = answer_space - a - b
    # An odd spare token may favor A by one token, not an entire response.
    extra_a = min(spare, la - a)
    a += extra_a
    b += min(spare - extra_a, lb - b)
    p += min(available - p - a - b, lp - p)
    assert p + a + b <= available
    return p, a, b


def assemble_input(p, a, b):
    budgets = allocate_budgets([len(p), len(a), len(b)])
    pieces = [trim_tokens(x, k) for x, k in zip([p, a, b], budgets)]
    ids = ([tokenizer.bos_token_id] + prefix_ids + pieces[0] + a_header_ids
           + pieces[1] + b_header_ids + pieces[2] + suffix_ids + [tokenizer.eos_token_id])
    assert len(ids) <= MAX_LENGTH
    return ids, [len(x) > k for x, k in zip([p, a, b], budgets)]

In [ ]:
# These cases were saved before training; no hidden-test labels are involved.
for case in read_json(ASSET_DIR / "tokenization_cases.json"):
    parts = [tokenizer.encode(case[k], add_special_tokens=False)
             for k in ["prompt_text", "response_a_text", "response_b_text"]]
    ids, _ = assemble_input(*parts)
    if ids != case["input_ids"]:
        raise RuntimeError(f"Training/inference tokenization differs for case {case['id']}.")
print("Exact training-time tokenization cases passed.")


def encode_frame(frame, description, swapped=False):
    records = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc=description):
        p, a, b = [parse_turns(v) for v in [row.prompt, row.response_a, row.response_b]]
        n = max(map(len, [p, a, b]))
        texts = [format_turns(ts + [""] * (n - len(ts))) for ts in [p, a, b]]
        if swapped:
            texts = [texts[0], texts[2], texts[1]]
        parts = [tokenizer.encode(text, add_special_tokens=False) for text in texts]
        ids, _ = assemble_input(*parts)
        records.append({"input_ids": ids, "attention_mask": [1] * len(ids)})
    return records


test_inputs = encode_frame(test_frame, "Tokenize test: original")
assert len(test_inputs) == len(test_frame)
print("Maximum encoded length:", max(len(x["input_ids"]) for x in test_inputs))

## 4. Repair the actual head-device failure
A copied hook can redirect an input *after* a PyTorch forward-pre-hook. Therefore this is not another outer
pre-hook patch. The saved classifier is reinstalled as an inference-only linear head with no inherited hooks,
using its saved FP32 weights exactly; normal Gemma last-non-padding-token pooling stays in Transformers.
Only local LoRA/head hooks are repaired. The model-wide hooks that transfer between backbone layer groups remain.

This in-memory model is for inference, not adapter switching or saving a new PEFT checkpoint. The source adapter
checkpoint and classification head on disk are unchanged.

In [ ]:
import torch.nn.functional as F
from collections import Counter
from accelerate.hooks import AlignDevicesHook, add_hook_to_module, remove_hook_from_module
from peft.tuners.lora.layer import LoraLayer


class DeviceSafeScore(torch.nn.Module):
    """Frozen saved classifier: align at the actual matrix multiplication, not an outer wrapper."""
    def __init__(self, saved_weight, device):
        super().__init__()
        if saved_weight.ndim != 2:
            raise ValueError("Classification weight must be a matrix.")
        self.out_features, self.in_features = saved_weight.shape
        # This is the trained tensor, not a random initialization or a merged LoRA weight.
        weight = saved_weight.detach().to(device=device, dtype=torch.float32).clone()
        self.weight = torch.nn.Parameter(weight, requires_grad=False)
        self.register_parameter("bias", None)

    def forward(self, hidden_states):
        # No inherited Accelerate/PEFT head hooks exist below this point.
        hidden_states = hidden_states.to(device=self.weight.device, dtype=self.weight.dtype)
        return F.linear(hidden_states, self.weight, self.bias)


def clear_local_device_hooks(module):
    """Remove direct Accelerate hooks bottom-up, including nested saved-head hooks.

    Check __dict__ instead of hasattr: PEFT wrappers can delegate attributes to a child.
    Bottom-up removal avoids mistaking that child's hook for the wrapper's own hook.
    """
    count = 0
    for child in reversed(list(module.modules())):
        if "_hf_hook" in child.__dict__ or "_old_forward" in child.__dict__:
            remove_hook_from_module(child, recurse=False)
            count += 1
    return count


def repair_inference_placement(peft_model, saved_weight):
    core = peft_model.get_base_model()
    devices = Counter()
    removed_lora_hooks = 0
    # Capture modules before changing their forward hooks. Never move the entire model.
    for name, layer in list(peft_model.named_modules()):
        if not isinstance(layer, LoraLayer):
            continue
        target = layer.get_base_layer().weight.device
        if target.type == "meta":
            raise RuntimeError(f"Unloaded LoRA base weight: {name}")
        removed_lora_hooks += clear_local_device_hooks(layer)
        layer.to(device=target)  # Preserve all dtypes and values, align only this local projection.
        if not all(p.device == target for p in layer.parameters()):
            raise RuntimeError(f"Adapter placement did not converge: {name}")
        add_hook_to_module(layer, AlignDevicesHook(
            execution_device=target, offload=False, io_same_device=False, place_submodules=True,
        ))
        devices[str(target)] += 1

    old_wrapper = core.score
    if not hasattr(old_wrapper, "modules_to_save") or "default" not in old_wrapper.modules_to_save:
        raise RuntimeError("Expected the trained PEFT classification-head wrapper.")
    old_weight = old_wrapper.modules_to_save["default"].weight.detach().cpu().float()
    torch.testing.assert_close(old_weight, saved_weight.cpu().float(), rtol=0, atol=0)
    old_head_device = str(old_wrapper.modules_to_save["default"].weight.device)
    removed_head_hooks = clear_local_device_hooks(old_wrapper)
    head_device = core.model.norm.weight.device
    if head_device.type == "meta":
        raise RuntimeError("Final normalization is still on meta.")
    core.score = DeviceSafeScore(saved_weight, head_device)
    torch.testing.assert_close(core.score.weight.detach().cpu(), saved_weight.cpu().float(), rtol=0, atol=0)
    assert all("_hf_hook" not in m.__dict__ and "_old_forward" not in m.__dict__
               for m in core.score.modules()), "The new head must be hook-free."
    del old_wrapper, old_weight
    return {"lora_layers_per_device": dict(devices), "removed_lora_hooks": removed_lora_hooks,
            "removed_head_hooks": removed_head_hooks, "previous_head_device": old_head_device,
            "head_device": str(head_device), "head_weight_exact": True,
            "head_implementation": "DeviceSafeScore (saved FP32 weights; no inherited hooks)"}

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU T4 x2 in Kaggle Session options.")
if "model" in globals() and model is not None:
    raise RuntimeError("A model is already loaded. Restart this Kaggle session rather than load a second copy.")
ngpu = torch.cuda.device_count()
for i in range(ngpu):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | free {free/2**30:.1f}/{total/2**30:.1f} GiB")

native_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
MODEL_DTYPE = torch.bfloat16 if native_bf16 else torch.float16
cfg = read_json(BASE_DIR / "config.json")
assert cfg["model_type"] == "gemma2" and cfg["num_hidden_layers"] == 42 and cfg["hidden_size"] == 3584
if ngpu >= 2:
    USED_GPUS = [0, 1]
    if any(torch.cuda.mem_get_info(i)[0] < 12 * 2**30 for i in USED_GPUS):
        raise RuntimeError("Less than 12 GiB is free on one GPU. Restart this Kaggle session to release stale models.")
    # Disjoint modules only: a root entry plus child overrides can temporarily
    # place the ENTIRE model on GPU 0 during Accelerate hook initialization.
    device_map = {
        "model.embed_tokens": 0,
        "model.rotary_emb": 0,
        **{f"model.layers.{i}": 0 if i < 19 else 1 for i in range(42)},
        "model.norm": 1,
        "score": 1,
    }
    # Use BF16 only if EVERY device used supports it natively.
    if not all(torch.cuda.get_device_capability(i)[0] >= 8 for i in USED_GPUS):
        MODEL_DTYPE = torch.float16
elif torch.cuda.mem_get_info(0)[0] >= 21.5 * 2**30:
    USED_GPUS = [0]
    device_map = {"": 0}
else:
    raise RuntimeError("A single 16 GB GPU is insufficient for this unquantized 9B model. Select T4 x2.")
for i in USED_GPUS:
    torch.cuda.set_per_process_memory_fraction(0.90, device=i)
print("Compute dtype:", MODEL_DTYPE, "| physical inference batch:", BATCH_SIZE)

adapter_path = ADAPTER_DIR / "adapter_model.safetensors"
ac = read_json(ADAPTER_DIR / "adapter_config.json")
assert ac["task_type"] == "SEQ_CLS" and ac["peft_type"] == "LORA" and "score" in ac["modules_to_save"]
assert int(read_json(ADAPTER_DIR / "trainer_state.json")["global_step"]) == 1440
with safe_open(str(adapter_path), framework="pt", device="cpu") as f:
    saved_head = f.get_tensor(manifest["head_key"]).float()
    adapter_keys = list(f.keys())
assert tuple(saved_head.shape) == (3, 3584) and torch.isfinite(saved_head).all()

base_model = AutoModelForSequenceClassification.from_pretrained(
    str(BASE_DIR), local_files_only=True, num_labels=3, problem_type="single_label_classification",
    id2label=dict(enumerate(LABEL_NAMES)), label2id={v:i for i,v in enumerate(LABEL_NAMES)},
    pad_token_id=tokenizer.pad_token_id, dtype=MODEL_DTYPE,
    attn_implementation="eager", device_map=device_map, low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
# Do not let PEFT deepcopy a head that still has a base-model dispatch hook.
clear_local_device_hooks(base_model.score)
base_model.score.float()
# Optional incompatible TorchAO was disabled BEFORE this call in section 2.
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False,
    autocast_adapter_dtype=True, low_cpu_mem_usage=True, local_files_only=True)

placement_report = repair_inference_placement(model, saved_head)
model.requires_grad_(False)
model.eval()
model.config.use_cache = False
model.gradient_checkpointing_disable()
core = model.get_base_model()
loaded_head = core.score.weight.detach().cpu().float()
torch.testing.assert_close(loaded_head, saved_head, rtol=0, atol=0)
assert sum("lora_B" in n for n, _ in model.named_parameters()) == sum("lora_B" in k for k in adapter_keys)
assert all(p.device.type == "cuda" for p in model.parameters()), "Unexpected CPU/meta offload."
assert any("lora_B" in n and torch.count_nonzero(p).item() > 0 for n, p in model.named_parameters())
INPUT_DEVICE = model.get_input_embeddings().weight.device
if len(USED_GPUS) == 2:
    assert core.model.layers[0].self_attn.q_proj.base_layer.weight.device.index == 0
    assert core.model.layers[41].self_attn.q_proj.base_layer.weight.device.index == 1
    assert core.score.weight.device.index == 1
for i in USED_GPUS:
    print(f"GPU {i} model allocation: {torch.cuda.memory_allocated(i) / 2**30:.2f} GiB")
(WORK_DIR / "device_placement.json").write_text(json.dumps(placement_report, indent=2))
print(json.dumps(placement_report, indent=2))
del saved_head, loaded_head, base_model, core
# Never call model.to('cuda:0') on this two-GPU model.
gc.collect()
print("Checkpoint-1440 loaded. Trained head is exact and hook-free.")

## 5. Test cross-device head routing before full inference
The first test sends small hidden-state tensors from **each GPU** into the repaired head and compares against
the same saved weight's direct matrix multiplication. This specifically exercises the failed path before test
prediction. The full-model smoke tests then check a long test sequence and a mixed-length padded test batch.

In [ ]:
@torch.inference_mode()
def check_head_device_routes():
    head = model.get_base_model().score
    assert isinstance(head, DeviceSafeScore)
    template = torch.linspace(-1, 1, steps=2 * head.in_features, dtype=torch.float32).reshape(2, head.in_features)
    checked = []
    for source in USED_GPUS:
        x = template.to(torch.device("cuda", source))
        with torch.autocast("cuda", dtype=MODEL_DTYPE):
            actual = head(x)
            expected = F.linear(x.to(device=head.weight.device, dtype=head.weight.dtype), head.weight)
        assert actual.device == head.weight.device
        torch.testing.assert_close(actual, expected, rtol=0, atol=0)
        assert torch.isfinite(actual).all()
        checked.append(f"cuda:{source} -> {head.weight.device}")
        del x, actual, expected
    print("Saved-head cross-device tests passed:", checked)


check_head_device_routes()

In [ ]:
@torch.inference_mode()
def predict_logits(inputs, description):
    if not inputs:
        return np.empty((0, 3), dtype=np.float32), 0.0
    lengths = np.asarray([len(x["input_ids"]) for x in inputs])
    order = np.argsort(lengths, kind="stable")
    logits = np.empty((len(inputs), 3), dtype=np.float32)
    position, bs = 0, int(BATCH_SIZE)
    started = time.perf_counter()
    with tqdm(total=len(inputs), desc=description) as progress:
        while position < len(order):
            idx = order[position:position + bs]
            batch = out = None
            oom = False
            try:
                batch = collator([inputs[int(i)] for i in idx])
                batch = {k: v.to(INPUT_DEVICE) for k, v in batch.items()}
                with torch.autocast("cuda", dtype=MODEL_DTYPE):
                    out = model(**batch, use_cache=False).logits
                if out.shape != (len(idx), 3) or not torch.isfinite(out).all():
                    raise RuntimeError("Non-finite/invalid logits. Do not submit; share this cell's output.")
                logits[idx] = out.float().cpu().numpy()
            except torch.OutOfMemoryError:
                if bs == 1:
                    raise
                oom = True
            finally:
                del batch, out
            if oom:
                bs = max(1, bs // 2)
                gc.collect()
                for device in USED_GPUS:
                    with torch.cuda.device(device):
                        torch.cuda.empty_cache()
                print("Retry this batch with physical batch", bs)
                continue
            position += len(idx)
            progress.update(len(idx))
    return logits, time.perf_counter() - started


def probabilities_from_logits(logits, temperature=1.0):
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be positive and finite.")
    z = np.asarray(logits, dtype=np.float64) / temperature
    if z.ndim != 2 or z.shape[1] != 3 or not np.isfinite(z).all():
        raise ValueError("Expected finite N-by-3 logits.")
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z)
    p /= p.sum(axis=1, keepdims=True)
    assert np.isfinite(p).all() and np.allclose(p.sum(1), 1.0)
    return p


def final_probabilities(original_logits, swapped_logits):
    if np.shape(original_logits) != np.shape(swapped_logits):
        raise ValueError("Original and swapped rows must be aligned.")
    temperature = FINAL_RECIPE["temperature"]
    original = probabilities_from_logits(original_logits, temperature)
    # Swapped A wins means the ORIGINAL B wins. Tie retains column 2.
    remapped = probabilities_from_logits(swapped_logits, temperature)[:, [1, 0, 2]]
    return 0.5 * (original + remapped)


# Only smoke tests run here; the full selection set is not loaded.
all_inputs = test_inputs
longest = int(np.argmax([len(x["input_ids"]) for x in all_inputs]))
shortest = int(np.argmin([len(x["input_ids"]) for x in all_inputs]))
_, _ = predict_logits([all_inputs[longest]], "Maximum-length model smoke test")
_, _ = predict_logits([all_inputs[shortest], all_inputs[longest]], "Mixed-length model smoke test")
del all_inputs
print("Full-model forward smoke tests passed.")

## 6. Predict all test rows with the final recipe
Both passes use the same checkpoint. A/B are swapped in the input for the second pass, then columns are remapped.
Apply `softmax(logits / 0.9)` to **each** pass and average probabilities, not logits. The 2B model is not loaded.
Only a completed, verified two-pass result becomes `submission.csv`.

In [ ]:
test_logits, test_original_seconds = predict_logits(test_inputs, "Predict test: original")
swapped_test_inputs = encode_frame(test_frame, "Tokenize test: swapped", swapped=True)
swapped_test_logits, test_swapped_seconds = predict_logits(swapped_test_inputs, "Predict test: swapped")
test_probs = final_probabilities(test_logits, swapped_test_logits)
assert test_probs.shape == (len(test_frame), 3)
test_seconds = test_original_seconds + test_swapped_seconds

In [ ]:
submission = pd.DataFrame(test_probs, columns=LABEL_COLUMNS)
submission.insert(0, "id", test_frame["id"].to_numpy())
# Preserve the sample submission's ID order if it contains the same runtime IDs.
sample_path = COMPETITION_DIR / "sample_submission.csv"
if sample_path.is_file():
    sample = pd.read_csv(sample_path, dtype={"id": str})
    assert list(sample.columns) == ["id", *LABEL_COLUMNS]
    if len(sample) == len(submission) and set(sample["id"]) == set(submission["id"]):
        submission = submission.set_index("id").loc[sample["id"]].reset_index()
assert len(submission) == len(test_frame) and submission["id"].is_unique
assert set(submission["id"]) == set(test_frame["id"])
values = submission[LABEL_COLUMNS].to_numpy()
assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
assert np.allclose(values.sum(1), 1.0, atol=1e-8)
output_path = WORK_DIR / "submission.csv"
temporary_path = WORK_DIR / "submission.csv.tmp"
submission.to_csv(temporary_path, index=False)
readback = pd.read_csv(temporary_path, dtype={"id": str})
assert readback["id"].equals(submission["id"])
assert list(readback.columns) == ["id", *LABEL_COLUMNS]
assert np.allclose(readback[LABEL_COLUMNS].sum(axis=1), 1.0, atol=1e-8)
# Publish the expected filename only after the complete CSV passes all checks.
os.replace(temporary_path, output_path)
report = {"checkpoint_step":1440, "base_revision":manifest["base_revision"],
    "adapter_sha256":manifest["adapter_sha256"], "rows":len(submission), "max_length":MAX_LENGTH,
    "dtype":str(MODEL_DTYPE), "devices":[torch.cuda.get_device_name(i) for i in USED_GPUS],
    "full_validation_performed":False, "test_seconds":test_seconds,
    "total_seconds":time.perf_counter()-NOTEBOOK_STARTED,
    "swap_tta":True, "temperature":0.9, "ensemble":False,
    "recipe":FINAL_RECIPE, "notebook_revision":NOTEBOOK_REVISION,
    "source_bundle_recipe":source_bundle_recipe, "placement":placement_report,
    "test_original_seconds":test_original_seconds, "test_swapped_seconds":test_swapped_seconds,
    "peak_allocated_gib":{str(i):torch.cuda.max_memory_allocated(i)/2**30 for i in USED_GPUS},
    "versions":{n:metadata.version(n) for n in ["torch","transformers","peft","accelerate"]}}
(WORK_DIR / "inference_report.json").write_text(json.dumps(report, indent=2))
print(submission.head().to_string(index=False))
print(f"\nWrote {len(submission):,} rows to {output_path}")
print("When Save & Run All succeeds, submit this SAVED VERSION's submission.csv to the competition.")
print("Kaggle re-runs it on the hidden test; a three-row development CSV is not the hidden-test result.")

## References and test scope

**Basis for checkpoint/recipe:** the uploaded completed `3_arena_gemma2_9b_memory_safe(2).ipynb`, specifically
its training summary and final comparison table. **Basis for the fix:** your attached failure log, which reaches
the saved head's inner Accelerate hook immediately before `F.linear` fails with cuda:0/cuda:1 weights/inputs.

Implementation references: [Accelerate hooks](https://huggingface.co/docs/accelerate/main/package_reference/big_modeling),
[PEFT saved-module wrapper](https://github.com/huggingface/peft/blob/main/src/peft/utils/other.py),
[Gemma sequence classification](https://huggingface.co/docs/transformers/model_doc/gemma2), and
[competition requirements](https://www.kaggle.com/competitions/llm-classification-finetuning).

This edit removes full selection-set validation and its data-loading/report dependencies from
`5_kaggle_9b_final_submission_fixed.ipynb`. The latest attached
`3_gemma2_9b_full_lora_submission.ipynb` was an older, original-order version without the head-device fix;
the previously fixed version is retained so that those repairs and the selected TTA/temperature recipe are not lost.

The notebook format, code syntax, removed validation references, unchanged model/device/preprocessing/TTA
functions, and synthetic smoke-test-to-CSV execution were checked for this edit.
**The edited notebook was not executed with the real 9B weights or on two physical T4 GPUs here.**
Smoke tests are retained, but full BF16/FP16 prediction-quality equivalence is not assessed by this submission notebook.